# 01 · PCAP → windowed traffic dataset (CSV / TXT)

Reads the captured `.pcapng` files, splits each capture into **5-second windows** and summarises every window as one sample: dominant source/destination, protocols, packet count, active ports, a human-readable payload summary (ARP / DNS / DHCP / raw) and a heuristic behaviour hint.

**Input:** `pcap_verileri/normal/*.pcapng`, `pcap_verileri/zararli/*.pcapng` (on Google Drive)
**Output:** `normal_dataset.csv` (117 windows), `zararli_dataset.csv` (872 windows) + `.txt` versions


In [ ]:
!pip install scapy

In [ ]:
import os
import pandas as pd
from scapy.all import rdpcap, IP, TCP, UDP, ARP, DNS, DHCP, BOOTP
from collections import Counter

def extract_advanced_payload(packet):
    """
    Farklı protokollere ait paketlerin içeriklerini (Payload)
    LLM'in anlayabileceği anlamlı metin özetlerine dönüştürür.
    """
    details = []

    # 1. ARP Katmanı Kontrolü
    if packet.haslayer(ARP):
        arp_layer = packet[ARP]
        if arp_layer.op == 1:
            details.append(f"ARP Request: Who has {arp_layer.pdst}? Tell {arp_layer.psrc}")
        elif arp_layer.op == 2:
            details.append(f"ARP Reply: {arp_layer.psrc} is at {arp_layer.hwsrc}")

    # 2. DNS Katmanı Kontrolü
    if packet.haslayer(DNS):
        dns_layer = packet[DNS]
        if dns_layer.qd:
            qname = dns_layer.qd.qname.decode('utf-8', errors='ignore')
            details.append(f"DNS Query: {qname}")
        if dns_layer.an:
            try:
                rdata = dns_layer.an.rdata
                details.append(f"DNS Answer resolved to: {rdata}")
            except:
                details.append("DNS Answer Present")

    # 3. DHCP / BOOTP Katmanı Kontrolü
    if packet.haslayer(DHCP):
        dhcp_options = packet[DHCP].options
        for opt in dhcp_options:
            if isinstance(opt, tuple) and opt[0] == 'message-type':
                dhcp_types = {1: "Discover", 2: "Offer", 3: "Request", 5: "ACK", 6: "NAK"}
                msg_type = dhcp_types.get(opt[1], f"Type_{opt[1]}")
                details.append(f"DHCP Message Type: {msg_type}")

    # 4. Ham Veri (Raw Payload) Kontrolü
    if packet.haslayer('Raw'):
        payload = packet['Raw'].load
        readable_text = "".join([chr(b) if 32 <= b < 127 or b in [10, 13] else "." for b in payload])
        if readable_text.strip():
            details.append(f"Raw Segment: {readable_text[:100].strip()}")

    if details:
        return " | ".join(details)
    return "No Layer Payload Data"

def process_pcap_folder(folder_path, label, time_window=5):
    """Belirtilen klasördeki tüm pcap/pcapng dosyalarını işler ve tek bir listede toplar."""
    folder_samples = []

    if not os.path.exists(folder_path):
        print(f"Hata: Klasör bulunamadı -> {folder_path}")
        return folder_samples

    # Klasördeki tüm dosyaları sırayla dön
    for file in os.listdir(folder_path):
        if file.endswith(('.pcap', '.pcapng')):
            pcap_path = os.path.join(folder_path, file)
            print(f"İşleniyor: {file} ({label})")

            try:
                packets = rdpcap(pcap_path)
            except Exception as e:
                print(f"Hata oluştu {file}: {e}")
                continue

            if not packets:
                continue

            start_time = float(packets[0].time)
            windows = {}

            # Paketleri kullanıcının istediği gibi 5'er saniyelik pencerelere bölme
            for pkt in packets:
                pkt_time = float(pkt.time)
                window_id = int((pkt_time - start_time) // time_window)

                if window_id not in windows:
                    windows[window_id] = []
                windows[window_id].append(pkt)

            for w_id, pkt_list in windows.items():
                packet_count = len(pkt_list)
                src_ips, dst_ips, protocols, ports, payload_summaries = [], [], [], [], []

                for pkt in pkt_list:
                    if pkt.haslayer(IP):
                        src_ips.append(pkt[IP].src)
                        dst_ips.append(pkt[IP].dst)
                        protocols.append(pkt[IP].proto)

                        if pkt.haslayer(TCP):
                            ports.append(pkt[TCP].sport)
                            ports.append(pkt[TCP].dport)
                        elif pkt.haslayer(UDP):
                            ports.append(pkt[UDP].sport)
                            ports.append(pkt[UDP].dport)

                    elif pkt.haslayer(ARP):
                        src_ips.append(pkt[ARP].psrc)
                        dst_ips.append(pkt[ARP].pdst)
                        protocols.append("ARP")

                    p_text = extract_advanced_payload(pkt)
                    if p_text != "No Layer Payload Data":
                        payload_summaries.append(p_text)

                most_common_src = Counter(src_ips).most_common(1)[0][0] if src_ips else "N/A"
                most_common_dst = Counter(dst_ips).most_common(1)[0][0] if dst_ips else "N/A"

                proto_counts = Counter(protocols)
                proto_list = []
                for p in proto_counts:
                    if p == 6 or p == "TCP": proto_list.append("TCP")
                    elif p == 17 or p == "UDP": proto_list.append("UDP")
                    elif p == 1 or p == "ICMP": proto_list.append("ICMP")
                    elif p == "ARP": proto_list.append("ARP")
                    else: proto_list.append(f"Proto_{p}")
                proto_summary = ", ".join(list(set(proto_list))) if proto_list else "Unknown"

                unique_ports = sorted(list(set(ports)))[:8]
                port_summary = ", ".join(map(str, unique_ports)) if unique_ports else "None"

                unique_payloads = list(set(payload_summaries))[:4]
                content_summary = " || ".join(unique_payloads) if unique_payloads else "No explicit layer payload detected."

                pattern = "General network communication."
                if "ARP Reply" in content_summary and packet_count > 10:
                    pattern = "Substantial ARP spoofing / MitM activities suspected."
                elif "DHCP Message Type" in content_summary:
                    pattern = "DHCP service traffic exchange (Potential DHCP Poisoning evaluation)."
                elif "DNS Query" in content_summary or "DNS Answer" in content_summary:
                    pattern = "DNS traffic lookup (Potential DNS Spoofing evaluation)."
                elif packet_count > 40 and len(set(ports)) > 10:
                    pattern = "High frequency port scanning patterns (Nmap SYN scan suspected)."

                sample = {
                    "Dosya Adi": file,
                    "Zaman Aralığı": f"{w_id * time_window}-{(w_id + 1) * time_window} sn",
                    "Baskın Kaynak": most_common_src,
                    "Baskın Hedef": most_common_dst,
                    "Gözlenen Protokoller": proto_summary,
                    "Toplam Paket Sayısı": packet_count,
                    "Aktif Portlar": port_summary,
                    "Paket İçerik Özetleri (Payload)": content_summary,
                    "Gözlemlenen Davranış Örüntüsü": pattern,
                    "Etiket": label
                }
                folder_samples.append(sample)

    return folder_samples

def save_to_txt(samples, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        for i, s in enumerate(samples, 1):
            f.write(f"--- TRAFİK ÖRNEĞİ {i} ---\n")
            f.write(f"Kaynak Dosya: {s['Dosya Adi']}\n")
            f.write(f"Zaman Aralığı: {s['Zaman Aralığı']}\n")
            f.write(f"Baskın İletişim: {s['Baskın Kaynak']} -> {s['Baskın Hedef']}\n")
            f.write(f"Protokoller: {s['Gözlenen Protokoller']}\n")
            f.write(f"Paket Miktarı: {s['Toplam Paket Sayısı']}\n")
            f.write(f"Port Hareketliliği: {s['Aktif Portlar']}\n")
            f.write(f"Paket İçerik Detayları (Payload): {s['Paket İçerik Özetleri (Payload)']}\n")
            f.write(f"Trafik Davranış Örüntüsü: {s['Gözlemlenen Davranış Örüntüsü']}\n")
            f.write(f"Sınıflandırma Etiketi: {s['Etiket']}\n")
            f.write("\n" + "="*50 + "\n\n")

# ---- DRIVE KLASÖR YOLLARI ----
pcap_klasor_normal = "/content/drive/MyDrive/pcap_verileri/normal/"
pcap_klasor_zararli = "/content/drive/MyDrive/pcap_verileri/zararli/"

# ---- 1. NORMAL TRAFİK DOSYALARINI AYRI İŞLE VE KAYDET ----
print("=== NORMAL KLASÖRÜ İŞLENİYOR ===")
normal_samples = process_pcap_folder(pcap_klasor_normal, label="normal", time_window=5)

if normal_samples:
    df_normal = pd.DataFrame(normal_samples)
    df_normal.to_csv("/content/drive/MyDrive/pcap_verileri/normal_dataset.csv", index=False, encoding='utf-8-sig')
    save_to_txt(normal_samples, "/content/drive/MyDrive/pcap_verileri/normal_dataset.txt")
    print(f">> Normal trafik için {len(df_normal)} adet 5 sn'lik örnek oluşturuldu.\n")

# ---- 2. ZARARLI TRAFİK DOSYALARINI AYRI İŞLE VE KAYDET ----
print("=== ZARARLI KLASÖRÜ İŞLENİYOR ===")
zararli_samples = process_pcap_folder(pcap_klasor_zararli, label="zararli", time_window=5)

if zararli_samples:
    df_zararli = pd.DataFrame(zararli_samples)
    df_zararli.to_csv("/content/drive/MyDrive/pcap_verileri/zararli_dataset.csv", index=False, encoding='utf-8-sig')
    save_to_txt(zararli_samples, "/content/drive/MyDrive/pcap_verileri/zararli_dataset.txt")
    print(f">> Zararlı trafik için {len(df_zararli)} adet 5 sn'lik örnek oluşturuldu.\n")

print("[İŞLEM TAMAMLANDI] Normal ve Zararlı veri kümeleri tamamen ayrıştırılarak Drive'a kaydedildi!")

=== NORMAL KLASÖRÜ İŞLENİYOR ===
İşleniyor: normal_trafik4.pcapng (normal)
İşleniyor: normal_trafik3.pcapng (normal)
İşleniyor: normal_trafik.pcapng (normal)
İşleniyor: normal_trafik2.pcapng (normal)
>> Normal trafik için 117 adet 5 sn'lik örnek oluşturuldu.

=== ZARARLI KLASÖRÜ İŞLENİYOR ===
İşleniyor: arp_ve_dns_saldirisi.pcapng (zararli)
İşleniyor: nmap_tarama.pcapng (zararli)
İşleniyor: dhcp_poisoning_trafigi.pcapng (zararli)
İşleniyor: arp_saldiri_trafigi.pcapng (zararli)
>> Zararlı trafik için 872 adet 5 sn'lik örnek oluşturuldu.

[İŞLEM TAMAMLANDI] Normal ve Zararlı veri kümeleri tamamen ayrıştırılarak Drive'a kaydedildi!
